In [1]:
# creating spark session
exec(open('/home/jovyan/.ipython/profile_default/startup/00-spark-session.py').read())

Spark 3.5.0 session ready as `spark` (Delta Lake enabled).


# Nth Highest Order Value Per Customer

## Difficulty
Medium

## Topics
- PySpark
- SQL
- Window Functions
- Ranking
- Filtering
- Partition By
- Row Number

## Problem Statement

You are given a PySpark DataFrame named `orders` containing customer order information.

### Dataset: `orders`

| Column | Data Type | Description |
|---|---|---|
| `order_id` | Integer | Unique identifier for each order |
| `customer_id` | Integer | Identifier of the customer |
| `amount` | Double | Order amount |
| `order_date` | String | Date of the order in `YYYY-MM-DD` format |

## Task

Find the **2nd highest order amount for each customer**.

### Requirements

1. Group orders by `customer_id`.
2. Rank each customer's orders based on `amount` in **descending order**.
3. Select the order having rank **2** for each customer.
4. If a customer has fewer than 2 orders, exclude that customer.
5. If multiple orders have the same amount, treat them as **distinct rows** for ranking purposes.
6. Therefore, use **row-based ranking**, not dense ranking.
7. Return only:
   - `customer_id`
   - `second_highest_amount`
8. Sort the final result by `customer_id` in **ascending order**.

## Important

For ties, each order must receive a separate position.

For example:

Customer 101:

| order_id | amount |
|---:|---:|
| 1 | 500.0 |
| 2 | 300.0 |
| 3 | 300.0 |
| 4 | 200.0 |

The row-based ranking would be:

| order_id | amount | rank |
|---:|---:|---:|
| 1 | 500.0 | 1 |
| 2 | 300.0 | 2 |
| 3 | 300.0 | 3 |
| 4 | 200.0 | 4 |

Therefore, the 2nd highest amount is `300.0`.

## Expected Output

| customer_id | second_highest_amount |
|---:|---:|
| 101 | 300.0 |
| 102 | 900.0 |

In [3]:
from pyspark.sql.functions import *

In [4]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType
)

orders_data = [
    (1, 101, 500.0, "2023-01-15"),
    (2, 101, 300.0, "2023-02-20"),
    (3, 101, 200.0, "2023-03-10"),
    (4, 102, 1000.0, "2023-01-05"),
    (5, 102, 750.0, "2023-02-15"),
    (6, 102, 900.0, "2023-03-25"),
    (7, 103, 50.0, "2023-01-01")
]

orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("amount", DoubleType(), False),
    StructField("order_date", StringType(), False)
])

orders = spark.createDataFrame(
    orders_data,
    orders_schema
)

orders.show()
orders.printSchema()

+--------+-----------+------+----------+
|order_id|customer_id|amount|order_date|
+--------+-----------+------+----------+
|       1|        101| 500.0|2023-01-15|
|       2|        101| 300.0|2023-02-20|
|       3|        101| 200.0|2023-03-10|
|       4|        102|1000.0|2023-01-05|
|       5|        102| 750.0|2023-02-15|
|       6|        102| 900.0|2023-03-25|
|       7|        103|  50.0|2023-01-01|
+--------+-----------+------+----------+

root
 |-- order_id: integer (nullable = false)
 |-- customer_id: integer (nullable = false)
 |-- amount: double (nullable = false)
 |-- order_date: string (nullable = false)



In [5]:
orders.createOrReplaceTempView("orders")

In [23]:
spark.sql("""
      with cte as (
              SELECT 
                  order_id,
                  amount,
                  customer_id,
                  row_number() over(partition by customer_id order by amount) as ranking
                  from orders
                  )
        select 
            ORDER_id,
            amount as second_highest_amount
            from cte 
            WHERE ranking = 2
        """).show()

+--------+---------------------+
|ORDER_id|second_highest_amount|
+--------+---------------------+
|       2|                300.0|
|       6|                900.0|
+--------+---------------------+



In [26]:
from pyspark.sql.window import Window 

window_specs = Window.partitionBy("customer_id").orderBy(col("amount").desc())

orders\
    .withColumn(
        "ranking",
        row_number().over(window_specs)
        )\
    .filter(
        col("ranking") == 2
        )\
    .select(
        col("order_id"),
        col("amount").alias("second_highest_amount")
    ).show()

+--------+---------------------+
|order_id|second_highest_amount|
+--------+---------------------+
|       2|                300.0|
|       6|                900.0|
+--------+---------------------+

